# Time-Series / Signal — โจทย์แบบ "สัญญาณ -> คลาส" (Classification)

หน้าต่างสัญญาณ 1 ช่วง -> 1 คลาส (เช่น EEG -> ระยะการนอน, ECG -> ชนิดการเต้นหัวใจ, เซนเซอร์ -> ท่าทาง)

วิธีในโน้ตบุ๊กนี้:
- วิธีที่ 1 (ง่ายสุด มือใหม่ทำแค่นี้) = สกัดฟีเจอร์ (พลังงานในแต่ละความถี่/สถิติ) + `AutoGluon`
- วิธีที่ 2 (ไม่บังคับ) = ฟีเจอร์เดิม + `LightGBM` แบ่ง CV ตาม subject (กันข้อมูลรั่ว)
- วิธีที่ 3 (ขั้นสูง ต้อง GPU) = `1D-CNN` บนสัญญาณดิบ (ดู reference_cheatsheet.md)


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ input เป็นสัญญาณตามเวลา (มี sampling rate) แบ่งเป็นหน้าต่าง แล้วทำนาย 1 คลาสต่อหน้าต่าง
ถ้าต้อง "ทำนายค่าตัวเลขในอนาคต" -> ไปใช้ `forecasting.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, `FS` (sampling rate), `LABEL_COL`, `GROUP_COL` (คอลัมน์ subject)

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install scipy lightgbm scikit-learn pandas numpy
!pip -q install -U "autogluon.tabular[all]"   # วิธีที่ 1

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "super-ai-engineer-ss-6-sleep-stage-classification"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

รองรับ CSV ที่แต่ละแถว = 1 หน้าต่าง คอลัมน์เป็นค่าสัญญาณ (เช่น s0..s2999) + คอลัมน์ label

In [ ]:
import os, pandas as pd, numpy as np
FS        = 100      # << แก้: จำนวนค่าต่อวินาที (Hz) -- EEG มัก 100, ECG มัก 250-500, accelerometer มือถือมัก 50 (ดูจากคำอธิบายข้อมูล)
LABEL_COL = "label"  # << แก้: ชื่อคอลัมน์คลาส -- ดูจาก display(tr.head()) แล้วเลือกคอลัมน์คำตอบ เช่น "stage", "target", "y"
GROUP_COL = None     # << แก้: คอลัมน์ที่บอกว่าแถวนี้มาจากคนไหน/อุปกรณ์ไหน เช่น "subject", "patient_id" -- ถ้ามีควรใส่ (กันคะแนนหลอก), ถ้าไม่มีใส่ None
TRAIN_CSV = os.path.join(DATA_DIR,"train.csv"); TEST_CSV=os.path.join(DATA_DIR,"test.csv")
SAMPLE_SUB= os.path.join(DATA_DIR,"sample_submission.csv")
tr=pd.read_csv(TRAIN_CSV); te=pd.read_csv(TEST_CSV); sample=pd.read_csv(SAMPLE_SUB)
ID_COL=sample.columns[0]; ANS_COL=sample.columns[1]
print("คอลัมน์ train:",list(tr.columns)[:8],"..."); display(tr.head())
drop=[c for c in [LABEL_COL,ID_COL,GROUP_COL] if c and c in tr.columns]
sig_cols=[c for c in tr.columns if c not in drop and pd.api.types.is_numeric_dtype(tr[c])]
Xtr_raw=tr[sig_cols].values; y_raw=tr[LABEL_COL].values
groups=tr[GROUP_COL].values if (GROUP_COL and GROUP_COL in tr.columns) else np.zeros(len(tr))
Xte_raw=te[[c for c in sig_cols if c in te.columns]].values
test_ids=te[ID_COL].values if ID_COL in te.columns else sample[ID_COL].values
classes=sorted(pd.unique(y_raw)); c2i={c:i for i,c in enumerate(classes)}; i2c={i:c for c,i in c2i.items()}
y=np.array([c2i[v] for v in y_raw]); N_CLASSES=len(classes)
print("จำนวนคลาส =",N_CLASSES,"| สัญญาณยาว",len(sig_cols),"ค่า/หน้าต่าง")

## สกัดฟีเจอร์ต่อหน้าต่าง (ใช้ทั้งวิธีที่ 1 และ 2)

In [ ]:
from scipy.signal import welch
from scipy.stats import skew, kurtosis
BANDS={"delta":(0.5,4),"theta":(4,8),"alpha":(8,13),"sigma":(11,16),"beta":(16,30),"gamma":(30,49)}
_trap=getattr(np,"trapezoid",np.trapz)   # numpy 2.0 เปลี่ยนชื่อ np.trapz -> np.trapezoid (รองรับทั้งสองเวอร์ชัน)
def feat(sig):
    sig=np.asarray(sig,dtype=np.float64)
    f={"mean":sig.mean(),"std":sig.std(),"min":sig.min(),"max":sig.max(),"ptp":np.ptp(sig),
       "skew":skew(sig),"kurt":kurtosis(sig),"rms":np.sqrt(np.mean(sig**2)),
       "zcr":np.mean(np.abs(np.diff(np.sign(sig))))/2}
    d1=np.diff(sig); d2=np.diff(d1); v0=sig.var()+1e-12; v1=d1.var()+1e-12; v2=d2.var()+1e-12
    f["hj_mob"]=np.sqrt(v1/v0); f["hj_comp"]=np.sqrt(v2/v1)/(np.sqrt(v1/v0)+1e-12)
    fr,psd=welch(sig,fs=FS,nperseg=min(len(sig),FS*2)); tot=_trap(psd,fr)+1e-12
    for nm,(lo,hi) in BANDS.items():
        idx=(fr>=lo)&(fr<hi); bp=_trap(psd[idx],fr[idx]) if idx.any() else 0.0
        f["bp_"+nm]=bp; f["rbp_"+nm]=bp/tot
    f["spec_ent"]=-np.sum((psd/tot)*np.log(psd/tot+1e-12)); return f
Xtr=pd.DataFrame([feat(w) for w in Xtr_raw]); Xte=pd.DataFrame([feat(w) for w in Xte_raw])
print("ฟีเจอร์:",Xtr.shape[1],"ตัว")

## 🔎 โจทย์นี้ต้องส่งอะไร + วัดด้วยอะไร (รันก่อนทำ submission)

เซลล์นี้ตอบ 3 คำถามที่มือใหม่มักไม่รู้:
- ต้องเติม "คอลัมน์อะไร" ลง submission และเป็น "ชนิดไหน" (ดูจาก sample_submission)
- โจทย์วัดด้วย "metric อะไร" (ดึงจาก Kaggle ให้อัตโนมัติ)
- ต้องส่งเป็น "ป้าย / ความน่าจะเป็น / ตัวเลข" แบบไหน

In [ ]:
# บอกว่าต้องเติมอะไรลง submission + ตัวอย่างค่าที่ sample ให้มา
print("ไฟล์ส่งต้องมีคอลัมน์:", list(sample.columns), "| จำนวนแถว:", len(sample))
for _c in list(sample.columns)[1:]:
    print(f"  - เติม '{_c}': ชนิด {sample[_c].dtype}, ตัวอย่างค่าใน sample = {list(sample[_c].dropna().unique()[:3])}")
# ดึง metric จาก Kaggle อัตโนมัติ (ถ้าต่อ API ได้)
_metric = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    _api = KaggleApi(); _api.authenticate()
    for _co in _api.competitions_list(search=COMP):
        if str(getattr(_co, "ref", "")).rstrip("/").split("/")[-1] == COMP:
            _metric = getattr(_co, "evaluationMetric", None) or getattr(_co, "evaluation_metric", None); break
except Exception:
    pass
def _how_to_send(m):
    m = (m or "").lower()
    if any(k in m for k in ["auc","roc","log loss","logloss","brier","probab"]): return "ความน่าจะเป็น (เลข 0-1)"
    if any(k in m for k in ["rmse","mae","mse","r2","rmsle","error","mape","smape"]): return "ตัวเลขต่อเนื่อง"
    return "ป้าย/คลาส (เช่น 0/1 หรือชื่อคลาส)"
print("\nMetric (ดึงจาก Kaggle):", _metric or "ดึงไม่ได้ -> เปิด tab Evaluation บนหน้า Kaggle อ่านเอง")
print("=> ปกติต้องส่งเป็น:", _how_to_send(_metric), "(เช็คกับ tab Evaluation อีกที)")

## วิธีที่ 1 — ฟีเจอร์ + AutoGluon (ง่ายสุด)

In [ ]:
from autogluon.tabular import TabularPredictor
ag=Xtr.copy(); ag[LABEL_COL]=y
pred=TabularPredictor(label=LABEL_COL,eval_metric="f1_macro",path="ag_sig").fit(
    ag, presets="best_quality", time_limit=600)   # << แก้ time_limit: วินาที (600=10นาที) ลอง 120 ก่อนให้รันผ่าน แล้วค่อยเพิ่มเป็น 1800-3600 ตอนส่งจริง
yhat=pred.predict(Xte).values
out=sample.copy(); out[ANS_COL]=[i2c[int(v)] for v in yhat]
out.to_csv("submission.csv",index=False); print("บันทึก submission.csv"); display(out.head())

## วิธีที่ 2 — ฟีเจอร์ + LightGBM แบ่ง CV ตาม subject (ไม่บังคับ แต่สำคัญถ้ามี subject)

แบ่ง CV ตาม subject เพื่อไม่ให้หน้าต่างจากคนเดียวกันอยู่ทั้ง train และ val (กันคะแนนหลอก)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
cls,cnt=np.unique(y,return_counts=True); cw={c:len(y)/(len(cls)*n) for c,n in zip(cls,cnt)}
sw=np.array([cw[v] for v in y])
params=dict(objective="multiclass",num_class=N_CLASSES,learning_rate=0.03,num_leaves=64,
            min_child_samples=40,n_jobs=-1,verbosity=-1)
oof=np.zeros((len(y),N_CLASSES)); tp=np.zeros((len(Xte),N_CLASSES))
# ถ้ามี subject หลายคน (>=5) แบ่ง CV ตาม subject (กันข้อมูลรั่ว) ; ถ้าไม่มี (GROUP_COL=None) ใช้ StratifiedKFold ปกติ ไม่งั้นจะ error
from sklearn.model_selection import StratifiedKFold
n_groups = len(np.unique(groups))
if n_groups >= 5:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(Xtr, y, groups)
else:
    print("ไม่มีคอลัมน์ subject (หรือ subject น้อยกว่า 5) -> ใช้ StratifiedKFold ธรรมดาแทน")
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(Xtr, y)
for tr_i,va_i in splitter:
    d=lgb.Dataset(Xtr.iloc[tr_i],y[tr_i],weight=sw[tr_i]); dv=lgb.Dataset(Xtr.iloc[va_i],y[va_i])
    m=lgb.train(params,d,2000,valid_sets=[dv],callbacks=[lgb.early_stopping(100),lgb.log_evaluation(0)])
    oof[va_i]=m.predict(Xtr.iloc[va_i]); tp+=m.predict(Xte)/5
print("OOF macro-F1 =", round(f1_score(y,oof.argmax(1),average="macro"),4))
out=sample.copy(); out[ANS_COL]=[i2c[int(v)] for v in tp.argmax(1)]
out.to_csv("submission_lgbm.csv",index=False); print("บันทึก submission_lgbm.csv")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "features autogluon signal"
!kaggle competitions submissions -c {COMP} | head